# MLP Probe Analysis — Multi-Turn Jailbreak Representations

This notebook trains MLP probes on hidden-state representations to study how
the model internally represents successful jailbreaks across three attack frameworks
(Crescendo, ActorAttack, XTeaming).

## Research questions

1. **In-framework:** Can a simple probe distinguish accepted harmful conversations
   from accepted harmless ones, using hidden states at the final turn? How does
   discriminability vary by layer?

2. **Cross-framework transfer:** If we train the probe on Crescendo jailbreaks and
   apply it to ActorAttack or XTeaming, does it still work? This tests whether
   jailbreak representations are framework-universal or framework-specific.

3. **K-sweep (Bullwinkel analog):** If we apply the final-turn-trained probe to
   *each individual turn* of a Crescendo conversation, does P(harmful) increase
   over turns? This would confirm that the harmfulness signal builds up progressively
   as the attack escalates — compliance priming visible in the probe's predictions.

## Token positions used

- **`h_inst`** — hidden state at the last user token (`t_inst`): encodes harmfulness
  of the *request*
- **`h_post`** — hidden state at the EOT tag (`t_post_inst`): encodes the model's
  compliance/refusal *decision*

## Classification task

`accepted_harmful` (y=1) vs `accepted_harmless` (y=0)

Both groups are conversations where the model complied. The probe must learn to
distinguish: "did the model comply with something harmful, or something benign?"
Refused conversations are excluded from training — a refusal looks different from
a successful jailbreak and would confuse the signal.

## Train / test split

- **Train:** attempt ≤ 16 (reps 1–16 per JBB goal)
- **Test:**  attempt > 16 (reps 17–20)
- **GroupKFold CV** by `pair_id` — the same JBB behavior never appears in both
  train and validation folds, so the probe cannot memorise topic signatures.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

REPR_ROOT   = repo_root / "data" / "representations"
FRAMEWORKS  = ["crescendo", "actorattack", "xteaming"]
N_LAYERS    = 32
HIDDEN_DIM  = 4096
TRAIN_MAX_ATTEMPT = 16
FOCAL_LAYER = 16   # 0-indexed → layer 17; same reference as nb04

FW_COLORS = {"crescendo": "#d62728", "actorattack": "#ff7f0e", "xteaming": "#2ca02c"}


In [ ]:
# Load trajectory data and assign behavioural categories + train/test split.
# Mirrors nb04 cells 2–3; self-contained so this notebook runs independently.

def assign_category(row):
    if row["goal_type"] == "harmful":
        return "accepted_harmful" if row["attack_success"] else "refused_harmful"
    else:
        return "accepted_harmless" if row["attack_success"] else "refused_harmless"

traj = {}
for fw in FRAMEWORKS:
    for split in ["harmful", "benign"]:
        folder = REPR_ROOT / "trajectories" / f"{fw}_{split}"
        meta   = pd.read_parquet(folder / "metadata.parquet")
        h_inst = np.load(folder / "h_inst.npy")
        h_post = np.load(folder / "h_post_inst.npy")
        meta["category"]   = meta.apply(assign_category, axis=1)
        meta["data_split"] = np.where(meta["attempt"] <= TRAIN_MAX_ATTEMPT, "train", "test")
        traj[(fw, split)]  = {"meta": meta, "h_inst": h_inst, "h_post": h_post}

print("Loaded. Class sizes per framework (train split):")
for fw in FRAMEWORKS:
    m_h = traj[(fw,"harmful")]["meta"]
    m_b = traj[(fw,"benign")]["meta"]
    tr_h = m_h[m_h["data_split"]=="train"]
    tr_b = m_b[m_b["data_split"]=="train"]
    n_ah  = tr_h[tr_h["category"]=="accepted_harmful"]["conversation_id"].nunique()
    n_ahl = tr_b[tr_b["category"]=="accepted_harmless"]["conversation_id"].nunique()
    print(f"  {fw:<14}  accepted_harmful={n_ah:4d}  accepted_harmless={n_ahl:4d}")


## Dataset construction

For each conversation we take the **final turn's** hidden state — one vector
per conversation per layer (shape `(32, 4096)`).

**Why final turn only?**
The final turn carries maximum attack context and is the point at which the
model has committed to its response. Using the final turn gives the clearest
signal and is the natural "Option A" centroid (see nb04).

**Why accepted only?**
We exclude `refused_harmful` and `refused_harmless` because the probe task is
*jailbreak detection* — distinguishing "the model just complied with something
harmful" from "the model just complied with something benign."
Refused conversations involve a different model state (the model is about to
decline) and would introduce a confound.

**Per-pair topic demeaning (`demean=True`, default).**
`accepted_harmful` (y=1) and `accepted_harmless` (y=0) come from different
JBB objectives (harmful vs benign), so the raw representations are confounded
by topic. Each JBB pair_id shares the same topic area across both trajectories,
so subtracting the per-pair mean (across y=1 and y=0 examples for that pair_id)
removes the topic-specific offset and forces the probe to learn what varies
*within* a topic — i.e., compliance state. Pair means computed on the training
split are cached for reuse when evaluating test data and k-sweep turns.

`build_dataset(framework, data_split, h_key, demean=True)` extracts:
- `X` — `(N, 32, 4096)` float32 — one representation per conversation
- `y` — binary labels (1 = accepted_harmful, 0 = accepted_harmless)
- `pair_ids` — JBB pair index for GroupKFold stratification


In [ ]:
# Cache per-pair means computed on the training split (reused for test + k-sweep)
_pair_means: dict = {}   # (framework, h_key) → {pair_id: (32, 4096) array}


def build_dataset(framework, data_split, h_key="h_inst", demean=True):
    """
    Extract final-turn representations for accepted_harmful (y=1)
    and accepted_harmless (y=0).

    demean=True: subtract per-pair mean (computed on train split) from each X,
    removing the topic-specific offset so the probe learns compliance state only.

    Returns
    -------
    X        : (N, 32, 4096) float32
    y        : (N,) int  —  1 = accepted_harmful, 0 = accepted_harmless
    pair_ids : (N,) int  —  JBB pair_id for GroupKFold
    """
    rows_X, rows_y, rows_p = [], [], []

    for goal_type, cat, label in [
        ("harmful", "accepted_harmful",  1),
        ("benign",  "accepted_harmless", 0),
    ]:
        meta  = traj[(framework, goal_type)]["meta"]
        h_arr = traj[(framework, goal_type)][h_key]

        mask = (meta["data_split"] == data_split) & (meta["category"] == cat)
        sub  = meta[mask]
        # Final turn per conversation
        idx  = sub.groupby("conversation_id")["turn_k"].idxmax()
        pos  = meta.index.get_indexer(idx.values)

        rows_X.append(h_arr[pos].astype(np.float32))
        rows_y.append(np.full(len(pos), label, dtype=int))
        rows_p.append(meta.loc[idx.values, "pair_id"].values)

    X        = np.concatenate(rows_X, axis=0)
    y        = np.concatenate(rows_y)
    pair_ids = np.concatenate(rows_p)

    if demean:
        cache_key = (framework, h_key)
        if data_split == "train":
            # Compute and cache per-pair means from training data
            pair_means = {}
            for pid in np.unique(pair_ids):
                mask = (pair_ids == pid)
                pair_means[pid] = X[mask].mean(axis=0)   # (32, 4096)
            _pair_means[cache_key] = pair_means
        else:
            pair_means = _pair_means.get(cache_key, {})

        for pid in np.unique(pair_ids):
            if pid in pair_means:
                mask = (pair_ids == pid)
                X[mask] -= pair_means[pid]

    return X, y, pair_ids


def demean_h(X_raw, pair_ids_raw, framework, h_key):
    """
    Apply cached per-pair means to arbitrary representations (e.g. k-sweep turns).
    Uses the pair means computed from the training split via build_dataset(..., demean=True).
    pair_ids_raw: array of pair_id for each row in X_raw.
    """
    cache_key = (framework, h_key)
    pair_means = _pair_means.get(cache_key, {})
    X = X_raw.copy()
    for pid in np.unique(pair_ids_raw):
        if pid in pair_means:
            mask = (pair_ids_raw == pid)
            X[mask] -= pair_means[pid]
    return X


# Sanity check (also populates _pair_means cache for train splits)
for fw in FRAMEWORKS:
    X, y, p = build_dataset(fw, "train")
    print(f"{fw}: train  X={X.shape}  y=1:{y.sum()}  y=0:{(y==0).sum()}  "
          f"unique_pairs={len(np.unique(p))}")
for fw in FRAMEWORKS:
    X, y, p = build_dataset(fw, "test")
    print(f"{fw}: test   X={X.shape}  y=1:{y.sum()}  y=0:{(y==0).sum()}")


## Probe design

**One probe per layer.** At each of the 32 layers we train an independent MLP
on the 4096-dim representation at that layer. This gives an AUROC curve vs layer
— directly comparable to the direction-based AUROC in nb04.

**Architecture:** `MLPClassifier(hidden_layer_sizes=(64,))` with early stopping
and StandardScaler normalisation. Small enough to avoid overfitting on ~1600
training examples, large enough to capture nonlinear structure if present.

**GroupKFold cross-validation** by `pair_id` (5 folds). Each fold holds out all
attempts for a subset of JBB behaviors, so the probe cannot memorise
"requests about chemistry are always harmful." CV gives an honest estimate of
goal-level generalisation within the training framework.

**Test evaluation:** After CV, refit on the full training set and evaluate on
the held-out test split (attempts 17–20).

**Two probes per framework:** one for `h_inst` and one for `h_post`, answering
different questions:
- `h_inst` probe: "does the model encode that the *request* is harmful?"
- `h_post` probe: "does the model's *generation-decision state* differ between
  harmful and benign compliance?"


In [ ]:
def train_probes(X_train, y_train, pair_ids, n_splits=5):
    """
    Train one MLP probe per layer with GroupKFold CV.

    Returns
    -------
    probes   : list of (StandardScaler, MLPClassifier) — one per layer
    cv_aucs  : (32,) mean CV AUROC per layer
    cv_stds  : (32,) std  CV AUROC per layer
    """
    gkf     = GroupKFold(n_splits=n_splits)
    probes  = []
    cv_aucs = []
    cv_stds = []

    for layer in range(N_LAYERS):
        X_l = X_train[:, layer, :]
        scaler = StandardScaler()
        X_ls   = scaler.fit_transform(X_l)

        fold_aucs = []
        for tr_idx, val_idx in gkf.split(X_ls, y_train, groups=pair_ids):
            if len(np.unique(y_train[val_idx])) < 2:
                continue
            clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=300,
                                random_state=42, early_stopping=True)
            clf.fit(X_ls[tr_idx], y_train[tr_idx])
            fold_aucs.append(roc_auc_score(
                y_train[val_idx], clf.predict_proba(X_ls[val_idx])[:, 1]))

        # Refit on full training set
        clf_full = MLPClassifier(hidden_layer_sizes=(64,), max_iter=300,
                                 random_state=42, early_stopping=True)
        clf_full.fit(X_ls, y_train)

        probes.append((scaler, clf_full))
        cv_aucs.append(np.mean(fold_aucs) if fold_aucs else np.nan)
        cv_stds.append(np.std(fold_aucs)  if fold_aucs else np.nan)

    return probes, np.array(cv_aucs), np.array(cv_stds)


def eval_probes(X_test, y_test, probes):
    """Return (32,) test AUROC array."""
    aucs = []
    for layer, (scaler, clf) in enumerate(probes):
        X_l = scaler.transform(X_test[:, layer, :])
        aucs.append(roc_auc_score(y_test, clf.predict_proba(X_l)[:, 1]))
    return np.array(aucs)


def probe_proba(X, probes, layer):
    """P(harmful) for every example in X at a single layer. Returns (N,)."""
    scaler, clf = probes[layer]
    X_l = scaler.transform(X[:, layer, :].astype(np.float32))
    return clf.predict_proba(X_l)[:, 1]


layers = np.arange(1, N_LAYERS + 1)
print("Probe functions defined.")


## Experiment 1 — In-framework probe: AUROC per layer

Train and evaluate within the same framework. Each subplot shows:
- **CV AUROC** (solid, ±1 std shaded): goal-stratified cross-validation on the
  training split — honest estimate of per-goal generalisation
- **Test AUROC** (dashed): held-out test split (attempts 17–20)

**Layout:** 2 rows (`h_inst` / `h_post`) × 3 columns (frameworks)

**What to look for:**
- Peak layer: where in the network does the probe perform best?
- Gap between CV and test: is the probe overfitting to training behaviors?
- `h_inst` vs `h_post`: which token position carries more discriminative information?


In [ ]:
# Train all in-framework probes (this will take a few minutes)
probe_results = {}   # (fw, h_key) → {"probes", "cv_aucs", "cv_stds", "test_aucs"}

for fw in FRAMEWORKS:
    for h_key in ["h_inst", "h_post"]:
        print(f"Training {fw} / {h_key} ...")
        X_tr, y_tr, pairs = build_dataset(fw, "train", h_key, demean=True)
        X_te, y_te, _     = build_dataset(fw, "test",  h_key, demean=True)
        probes, cv_aucs, cv_stds = train_probes(X_tr, y_tr, pairs)
        test_aucs = eval_probes(X_te, y_te, probes)
        probe_results[(fw, h_key)] = {
            "probes": probes, "cv_aucs": cv_aucs,
            "cv_stds": cv_stds, "test_aucs": test_aucs,
        }
        best = int(np.nanargmax(test_aucs))
        print(f"  peak test AUROC = {test_aucs[best]:.3f} at layer {best+1}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, fw in enumerate(FRAMEWORKS):
    for row, h_key in enumerate(["h_inst", "h_post"]):
        ax = axes[row, col]
        r  = probe_results[(fw, h_key)]

        ax.plot(layers, r["cv_aucs"],   color="#1f77b4", lw=1.5, label="CV AUROC")
        ax.fill_between(layers,
                        r["cv_aucs"] - r["cv_stds"],
                        r["cv_aucs"] + r["cv_stds"],
                        color="#1f77b4", alpha=0.15)
        ax.plot(layers, r["test_aucs"], color="#d62728", lw=1.5,
                ls="--", label="Test AUROC")
        ax.axhline(0.5, color="gray", lw=0.5, ls="--")
        ax.set_ylim(0.4, 1.05)
        ax.set_xlabel("Layer")
        ax.set_title(f"{fw} — {h_key}")
        if col == 0:
            ax.set_ylabel("AUROC  (accepted_harmful vs accepted_harmless)")
        ax.legend(fontsize=8)

plt.suptitle("In-framework MLP probe AUROC per layer\n"
             "(accepted_harmful vs accepted_harmless, final turn)", y=1.01)
plt.tight_layout()
plt.savefig("../figures/05_infw_auroc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/05_infw_auroc.png")


## Experiment 2 — Cross-framework transfer

Train the probe on framework A (train split), evaluate on framework B (test split).
This tests whether the hidden-state signature of a successful jailbreak is
**framework-universal** or specific to how each attack strategy phrases requests.

**3×3 heatmaps** — rows = train framework, columns = test framework:
- **Diagonal:** in-distribution performance (same as Experiment 1 best layer)
- **Off-diagonal:** transfer — does a Crescendo-trained probe detect ActorAttack
  jailbreaks?

We show results at:
1. The **best layer** of the train-framework probe (maximising in-framework test AUROC)
2. **Layer 17** (focal, same reference as nb04) for a fixed comparison point

Strong off-diagonal performance would mean jailbreak representations are
framework-agnostic in the model's hidden states.


In [ ]:
# For each (train_fw, test_fw, h_key): evaluate the train_fw probe on test_fw data
# at (a) the best layer for train_fw and (b) FOCAL_LAYER

transfer = {}   # (h_key, "best"/"focal") → (3, 3) AUROC matrix

for h_key in ["h_inst", "h_post"]:
    mat_best  = np.zeros((3, 3))
    mat_focal = np.zeros((3, 3))

    for i, train_fw in enumerate(FRAMEWORKS):
        probes    = probe_results[(train_fw, h_key)]["probes"]
        best_l    = int(np.nanargmax(probe_results[(train_fw, h_key)]["test_aucs"]))

        for j, test_fw in enumerate(FRAMEWORKS):
            X_te, y_te, _ = build_dataset(test_fw, "test", h_key, demean=True)
            mat_best[i, j]  = roc_auc_score(y_te, probe_proba(X_te, probes, best_l))
            mat_focal[i, j] = roc_auc_score(y_te, probe_proba(X_te, probes, FOCAL_LAYER))

    transfer[(h_key, "best")]  = mat_best
    transfer[(h_key, "focal")] = mat_focal

# ── Plot: 2×2 grid (rows=h_key, cols=best/focal) ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

configs = [
    (axes[0,0], "h_inst", "best",  "h_inst — best layer"),
    (axes[0,1], "h_inst", "focal", f"h_inst — layer {FOCAL_LAYER+1}"),
    (axes[1,0], "h_post", "best",  "h_post — best layer"),
    (axes[1,1], "h_post", "focal", f"h_post — layer {FOCAL_LAYER+1}"),
]

for ax, h_key, which, title in configs:
    mat = transfer[(h_key, which)]
    im  = ax.imshow(mat, vmin=0.5, vmax=1.0, cmap="RdYlGn")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(FRAMEWORKS, rotation=30, ha="right")
    ax.set_yticklabels(FRAMEWORKS)
    ax.set_xlabel("Test framework"); ax.set_ylabel("Train framework")
    ax.set_title(title)
    for ii in range(3):
        for jj in range(3):
            val = mat[ii, jj]
            ax.text(jj, ii, f"{val:.3f}", ha="center", va="center",
                    fontsize=10, color="black" if val < 0.85 else "white")
    plt.colorbar(im, ax=ax)

plt.suptitle("Cross-framework MLP probe transfer\n"
             "(accepted_harmful vs accepted_harmless)", y=1.01)
plt.tight_layout()
plt.savefig("../figures/05_crossfw_transfer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/05_crossfw_transfer.png")


## Experiment 3 — K-sweep: does P(harmful) build up over turns?

This is the direct analog of Bullwinkel et al. Figure 2, using `h_inst` and
`h_post` instead of response tokens.

**Setup:** Take the probe trained on the **final turn** of each framework.
Apply it — without any retraining — to the hidden states at **each individual
turn k** of test conversations (k = 1, 2, ..., n).

At turn k the model has seen only k turns of context. We record mean P(harmful)
across all test conversations at that turn.

**Expected result:**
- `accepted_harmful` P(harmful) should **increase** over turns — as the attack
  escalates the user messages become increasingly harmful-looking to the probe
- `accepted_harmless` P(harmful) should stay **flat and low** — benign
  conversations don't escalate

A rising curve for accepted_harmful and a flat curve for accepted_harmless
confirms compliance priming is visible in the probe's predictions: the model's
internal representation of each successive user message shifts progressively
toward "harmful" as the attack context accumulates.

### In-framework k-sweep
Each framework's probe is applied to its own test conversations.

### Cross-framework k-sweep
The Crescendo probe is applied to ActorAttack and XTeaming test conversations
turn-by-turn. If P(harmful) still rises, the Crescendo-trained probe detects
the build-up in other frameworks — the harmfulness signal is universal.


In [ ]:
# In-framework k-sweep: apply each framework's probe to its own per-turn test data.
# Use the best layer of the h_inst probe for each framework.

MIN_CONVS = 5   # minimum conversations at turn k to include that point

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, fw in enumerate(FRAMEWORKS):
    best_layer = int(np.nanargmax(probe_results[(fw, "h_inst")]["test_aucs"]))

    for row, h_key in enumerate(["h_inst", "h_post"]):
        ax     = axes[row, col]
        probes = probe_results[(fw, h_key)]["probes"]

        for cat, goal_type, color, label in [
            ("accepted_harmful",  "harmful", "#d62728", "accepted harmful"),
            ("accepted_harmless", "benign",  "#1f77b4", "accepted harmless"),
        ]:
            meta  = traj[(fw, goal_type)]["meta"]
            h_arr = traj[(fw, goal_type)][h_key]
            test_m = meta[meta["data_split"] == "test"]

            max_k  = int(test_m[test_m["category"] == cat]["turn_k"].max())                      if (test_m["category"] == cat).any() else 0

            means, sems, turn_ns = [], [], []
            for k in range(1, max_k + 1):
                rows_k = test_m[(test_m["category"] == cat) & (test_m["turn_k"] == k)]
                if len(rows_k) < MIN_CONVS:
                    break
                pos  = meta.index.get_indexer(rows_k.index)
                X_k  = h_arr[pos].astype(np.float32)
                pids = rows_k["pair_id"].values
                X_k  = demean_h(X_k, pids, fw, h_key)
                p    = probe_proba(X_k, probes, best_layer)
                means.append(p.mean())
                sems.append(p.std() / np.sqrt(len(p)))
                turn_ns.append(k)

            if means:
                means = np.array(means); sems = np.array(sems)
                ax.plot(turn_ns, means, color=color, marker="o", ms=4, label=label)
                ax.fill_between(turn_ns, means-sems, means+sems, color=color, alpha=0.2)

        ax.axhline(0.5, color="gray", lw=0.5, ls="--")
        ax.set_ylim(0.0, 1.05)
        ax.set_xlabel("Turn k")
        pos_name = "h_inst (request)" if h_key == "h_inst" else "h_post (decision)"
        ax.set_title(f"{fw} — {pos_name}\n(probe layer {best_layer+1})")
        if col == 0:
            ax.set_ylabel("Mean P(harmful)")
        ax.legend(fontsize=8)

plt.suptitle("In-framework k-sweep: mean P(harmful) over turns\n"
             "(probe trained on final-turn data, applied turn-by-turn)", y=1.01)
plt.tight_layout()
plt.savefig("../figures/05_ksweep_infw.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/05_ksweep_infw.png")

In [ ]:
# Cross-framework k-sweep: train probe on Crescendo, apply to all 3 frameworks.
# This tests whether Crescendo-trained probe detects build-up in other frameworks.

TRAIN_FW   = "crescendo"
best_layer = int(np.nanargmax(probe_results[(TRAIN_FW, "h_inst")]["test_aucs"]))
print(f"Using {TRAIN_FW} h_inst probe, best layer = {best_layer+1}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for col, test_fw in enumerate(FRAMEWORKS):
    ax = axes[col]

    for cat, goal_type, color, label in [
        ("accepted_harmful",  "harmful", "#d62728", "accepted harmful"),
        ("accepted_harmless", "benign",  "#1f77b4", "accepted harmless"),
    ]:
        meta   = traj[(test_fw, goal_type)]["meta"]
        h_arr  = traj[(test_fw, goal_type)]["h_inst"]
        test_m = meta[meta["data_split"] == "test"]
        probes = probe_results[(TRAIN_FW, "h_inst")]["probes"]

        max_k  = int(test_m[test_m["category"] == cat]["turn_k"].max())                  if (test_m["category"] == cat).any() else 0

        means, sems, turn_ns = [], [], []
        for k in range(1, max_k + 1):
            rows_k = test_m[(test_m["category"] == cat) & (test_m["turn_k"] == k)]
            if len(rows_k) < MIN_CONVS:
                break
            pos = meta.index.get_indexer(rows_k.index)
            X_k = h_arr[pos].astype(np.float32)
            pids = rows_k["pair_id"].values
            X_k  = demean_h(X_k, pids, TRAIN_FW, "h_inst")
            p   = probe_proba(X_k, probes, best_layer)
            means.append(p.mean())
            sems.append(p.std() / np.sqrt(len(p)))
            turn_ns.append(k)

        if means:
            means = np.array(means); sems = np.array(sems)
            ax.plot(turn_ns, means, color=color, marker="o", ms=4, label=label)
            ax.fill_between(turn_ns, means-sems, means+sems, color=color, alpha=0.2)

    ax.axhline(0.5, color="gray", lw=0.5, ls="--")
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel("Turn k")
    ax.set_title(f"Test: {test_fw}\n(probe trained on {TRAIN_FW})")
    ax.set_ylabel("Mean P(harmful)")
    ax.legend(fontsize=8)

plt.suptitle(f"Cross-framework k-sweep — {TRAIN_FW} probe → all frameworks\n"
             f"(h_inst, layer {best_layer+1})", y=1.01)
plt.tight_layout()
plt.savefig("../figures/05_ksweep_crossfw.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/05_ksweep_crossfw.png")


## Summary

The table below collects peak test AUROC and best layer for each
(framework, token position) combination from Experiment 1.

**Reading the results across experiments:**

| Result | Implication |
|---|---|
| High in-framework AUROC | The model encodes a detectable jailbreak signal at that token position |
| High off-diagonal transfer (Exp 2) | The jailbreak representation is framework-universal |
| Rising P(harmful) in k-sweep (Exp 3) | Compliance priming is visible in the probe's predictions over turns |
| P(harmless) flat in k-sweep | The build-up is specific to harmful conversations, not a general multi-turn effect |


In [ ]:
rows = []
for fw in FRAMEWORKS:
    for h_key in ["h_inst", "h_post"]:
        r = probe_results[(fw, h_key)]
        best_l = int(np.nanargmax(r["test_aucs"]))
        rows.append({
            "framework":        fw,
            "position":         h_key,
            "peak_cv_auc":      float(np.nanmax(r["cv_aucs"])),
            "peak_test_auc":    float(np.nanmax(r["test_aucs"])),
            "best_layer":       best_l + 1,
            "focal_layer_auc":  float(r["test_aucs"][FOCAL_LAYER]),
        })

df = pd.DataFrame(rows).set_index(["framework","position"]).round(3)
display(df)
